# GRM shard run (production)

Computes the real sharded GRM (`plink --make-grm-bin --parallel k N_SHARDS`
across all `k`), on this interactive VM, no `dsub`/Batch involved.

Per-shard cost is flat regardless of row range -- fixed cost (loading the
full panel) dominates -- and peak memory per shard is small (<1 GB) once
`--read-freq`/`--memory` are applied, so concurrency is CPU-bound, not
RAM-bound, on this `n1-highmem-64` machine. But `--make-grm-bin` is *not*
single-threaded (contrary to plink's own docs) and per-thread efficiency
drops fast as `--threads` increases (measured: 65% efficient at 4 threads,
34% at 12) -- see the Run Parameters cell for the Amdahl's-law fit behind
`N_CONCURRENT`/`N_THREADS`. At `N_SHARDS=300`, current settings estimate
**~8.2 hours** wall-clock -- a real interactive-VM session length, not the
originally-hoped ~2 hours, but still far cheaper than any Batch/dsub
alternative once redundant per-task panel downloads are counted (see
`grm_shard_batch_submit.ipynb`'s Status notes and the cost comparison
worked out for the 2026-07-19/20 validation batch).

Plink only -- no `GRM-pairs`/`grm_shard_tool` build or code path touched
here; that's a separate later step for the phenotype cross-product.

## Setup

In [ ]:
%%bash
set -e

BIN_DIR="$HOME/bin"
mkdir -p "$BIN_DIR"

if [ ! -x "$BIN_DIR/plink" ]; then
  PLINK_URL="https://s3.amazonaws.com/plink1-assets/plink_linux_x86_64_20230116.zip"
  cd /tmp
  wget -q -O plink.zip "$PLINK_URL"
  unzip -o -q plink.zip plink -d "$BIN_DIR"
  chmod +x "$BIN_DIR/plink"
fi

export PATH="$BIN_DIR:$PATH"
plink --version
nproc

In [ ]:
import os

bin_dir = os.path.expanduser("~/bin")
if bin_dir not in os.environ["PATH"].split(":"):
    os.environ["PATH"] = f"{bin_dir}:{os.environ['PATH']}"


## Paths

Same layout as `grm_shard_timing.ipynb` -- local scratch copy of the
genome-wide bed/bim/fam, precomputed allele frequencies. Recomputes
`FREQ_PATH`/`MEMORY_MB` rather than assuming the timing notebook's kernel
is still alive with those variables in memory.

In [ ]:
WORKSPACE_BUCKET = os.path.expanduser(
    "~/workspace/Data from All of Us Controlled Tier /shared-env-pilot"
)
BUCKET_DIR = f"{WORKSPACE_BUCKET}/data/03_grm_shards"

LOCAL_WORK_DIR = os.path.expanduser("~/scratch_grm")
SHARD_OUT_DIR = os.path.join(LOCAL_WORK_DIR, "shards")
os.makedirs(SHARD_OUT_DIR, exist_ok=True)

BED_NAME = "genome_wide_round2b_thinned_bed"   # from genome_wide_qc_thinning_merge.ipynb
BED_PREFIX = os.path.join(LOCAL_WORK_DIR, BED_NAME)

for ext in ("bed", "bim", "fam"):
    bucket_path = f"{BUCKET_DIR}/{BED_NAME}.{ext}"
    local_path = f"{BED_PREFIX}.{ext}"
    assert os.path.isfile(bucket_path), f"missing merged bed export: {bucket_path!r} -- run genome_wide_qc_thinning_merge.ipynb's merge section first"
    if not os.path.isfile(local_path):
        import shutil
        shutil.copy(bucket_path, local_path)

print(BED_PREFIX)

In [ ]:
%%bash -s "$BED_PREFIX"
set -e
BED_PREFIX=$1

FREQ_PATH="${BED_PREFIX}_freq.frq"
if [ ! -f "$FREQ_PATH" ]; then
  plink --bfile "$BED_PREFIX" --freq --out "${BED_PREFIX}_freq"
fi

In [ ]:
FREQ_PATH = f"{BED_PREFIX}_freq.frq"
assert os.path.isfile(FREQ_PATH), f"missing precomputed frequencies: {FREQ_PATH!r}"

BED_SIZE_GB = os.path.getsize(f"{BED_PREFIX}.bed") / 1e9
MEMORY_MB = int((BED_SIZE_GB * 2 + 4) * 1024)   # same sizing as grm_shard_timing.ipynb

print(f"FREQ_PATH = {FREQ_PATH}")
print(f"Bed file size: {BED_SIZE_GB:.1f} GB -> --memory {MEMORY_MB} MB")

## Run parameters

`N_SHARDS`: settled on 300 (low end of the 200-500 range considered,
since per-shard cost doesn't shrink with fewer shards -- see
`grm_shard_timing.ipynb`). `N_CONCURRENT`/`N_THREADS`: derived from real
single-shard timing at two different `--threads` settings (not from the
RAM/CPU-bound feasibility numbers alone, which only bounded max
concurrency -- they didn't account for `--make-grm-bin`'s own internal,
inefficient multi-threading). See the cell below for the fit and the
comparison that picked these specific values.

In [ ]:
N_SHARDS = 300      # per-shard cost is flat (~809s, 2% spread) regardless of N_SHARDS, so
                    # pick the low end of the 200-500 range -- more shards is pure overhead

# Isolated single-shard timing (no concurrent contention) at two thread counts:
#   --threads 4:  1473.3s
#   --threads 12:  958.6s
# Fitting Amdahl's law (T(n) = S + P/n) to those two points gives S=701.3s (serial floor),
# P=3088.2 (thread-seconds of parallel work) -- i.e. per-thread efficiency drops fast (65%
# at 4 threads, 34% at 12), so total wall-clock for N_SHARDS shards (= waves * T(threads),
# where waves = ceil(N_SHARDS / N_CONCURRENT) and N_CONCURRENT * N_THREADS stays within the
# vCPU budget) is minimized by MORE concurrent shards at FEWER threads each, not the other
# way around.
#
# Real production run at (N_CONCURRENT=15, N_THREADS=4) confirmed the 65%-efficiency
# figure directly: top showed each process using ~230-320% CPU (avg ~2.8 cores) against
# the requested 4 -- meaning 15 concurrent processes only demanded ~42 cores, leaving
# ~34% of the 64-core machine idle (load average ~42, %Cpu(s) showed 33.9% idle, no
# iowait, no swap). Bumped N_CONCURRENT to make use of that headroom: 64 / 2.8 ~= 22.9,
# rounded down to 20 for margin (some processes ran above the 2.8 average, up to ~3.2).
#   (N_CONCURRENT=15, N_THREADS=4):  20 waves * 1473.3s ~= 8.2 hours
#   (N_CONCURRENT=20, N_THREADS=4):  15 waves * 1473.3s ~= 6.1 hours (assuming no new
#     contention appears -- unverified above 15-way concurrency, worth watching top
#     again after bumping this in case efficiency drops further at higher concurrency)
N_CONCURRENT = 20
N_THREADS = 4

In [ ]:
%%bash -s "$BED_PREFIX" "$FREQ_PATH" "$MEMORY_MB" "$N_THREADS" "$N_SHARDS" "$SHARD_OUT_DIR"
set -e
BED_PREFIX=$1
FREQ_PATH=$2
MEMORY_MB=$3
N_THREADS=$4
N_SHARDS=$5
SHARD_OUT_DIR=$6

# single-shard timing check at the real N_THREADS setting, before committing to the full
# N_CONCURRENT x N_SHARDS run -- k=1 is as representative as any other k (grm_shard_timing.ipynb
# confirmed per-shard cost is flat regardless of row range)
TIME_LOG=$(mktemp)
TIMEFORMAT='%R %U %S'
{ time plink --bfile "$BED_PREFIX" --make-grm-bin \
  --parallel 1 "$N_SHARDS" \
  --read-freq "$FREQ_PATH" --memory "$MEMORY_MB" \
  --threads "$N_THREADS" \
  --out "${SHARD_OUT_DIR}/grm_shard_1_of_${N_SHARDS}_test"; } 2>> "$TIME_LOG"

# TIMEFORMAT's line is appended last, after any of plink's own stderr output --
# only the final line matters
read -r REAL_SEC USER_SEC SYS_SEC < <(tail -1 "$TIME_LOG")
python3 -c "
real, user = $REAL_SEC, $USER_SEC
ratio = user / real if real else float('nan')
print(f'real={real:.1f}s  user={user:.1f}s  user/real ratio={ratio:.2f}x '
      f'(effective parallelism achieved out of --threads $N_THREADS requested)')
"
rm -f "$TIME_LOG"

## Driver

Same `subprocess` + `ThreadPoolExecutor` pattern as
`genome_wide_qc_thinning_merge.ipynb`'s chromosome driver -- there it was
capped by vCPU count directly since each `plink2` call was itself
multi-threaded; here each `plink --make-grm-bin` shard is single-process
and the cap comes from `grm_shard_timing.ipynb` instead (RAM- or
CPU-bound, whichever binds). One call per `k` in `range(1, N_SHARDS + 1)`,
same `--read-freq`/`--memory` flags as the timing run.

In [ ]:
import subprocess
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

def run_shard(k, n_shards):
    out_prefix = os.path.join(SHARD_OUT_DIR, f"grm_shard_{k}_of_{n_shards}")
    # plink's real --parallel output naming is "<out>.grm.bin.<k>" (confirmed from real
    # run logs), not "<out>.grm.bin" -- check both a local copy (from an earlier partial
    # run of this same notebook) and an already-persisted bucket copy (from a previous
    # full run) before recomputing, so a re-run of this cell after an interruption picks
    # up where it left off instead of redoing already-finished shards.
    local_bin = f"{out_prefix}.grm.bin.{k}"
    bucket_bin = os.path.join(BUCKET_DIR, "grm_shards", f"grm_shard_{k}_of_{n_shards}.grm.bin.{k}")

    for existing_path, location in ((local_bin, "local"), (bucket_bin, "bucket")):
        if os.path.isfile(existing_path) and os.path.getsize(existing_path) > 0:
            return {
                "k": k, "elapsed_sec": 0.0, "returncode": 0,
                "out_prefix": out_prefix, "stdout_tail": None, "skipped": location,
            }

    cmd = ["plink", "--bfile", BED_PREFIX, "--make-grm-bin",
           "--parallel", str(k), str(n_shards),
           "--read-freq", FREQ_PATH, "--memory", str(MEMORY_MB),
           "--threads", str(N_THREADS),
           "--out", out_prefix]

    start = time.monotonic()
    proc = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    elapsed_sec = time.monotonic() - start

    return {
        "k": k, "elapsed_sec": elapsed_sec, "returncode": proc.returncode,
        "out_prefix": out_prefix,
        "stdout_tail": proc.stdout[-500:] if proc.returncode != 0 else None,
        "skipped": None,
    }

results = []
start_all = time.monotonic()
with ThreadPoolExecutor(max_workers=N_CONCURRENT) as pool:
    futures = {pool.submit(run_shard, k, N_SHARDS): k for k in range(1, N_SHARDS + 1)}
    for i, future in enumerate(as_completed(futures), 1):
        r = future.result()
        results.append(r)
        if r["skipped"]:
            status = f"skipped (already in {r['skipped']})"
        elif r["returncode"] == 0:
            status = "ok"
        else:
            status = "FAILED (see stdout_tail)"
        print(f"[{i}/{N_SHARDS}] k={r['k']:>4}: {r['elapsed_sec']:.1f}s, {status}")

wall_clock_hours = (time.monotonic() - start_all) / 3600
n_skipped = sum(1 for r in results if r["skipped"])
print(f"\nAll shards done in {wall_clock_hours:.2f} hours wall-clock ({n_skipped}/{N_SHARDS} skipped, already present)")

failed = [r for r in results if r["returncode"] != 0]
if failed:
    print(f"\n{len(failed)} shard(s) FAILED:")
    for r in failed:
        print(f"  k={r['k']}: {r['stdout_tail']}")

## Persist to bucket

Only one `.grm.id` is kept -- every shard sees the identical sample panel
(no per-shard `--keep`), so every shard's `.grm.id` is byte-identical.
Keeping all `N_SHARDS` copies would just be `N_SHARDS - 1` redundant
copies of the same small file; one canonical copy (from `k=1`) is enough
for `grm_shard_tool`'s downstream ID lookup.

In [ ]:
%%bash -s "$SHARD_OUT_DIR" "$BUCKET_DIR" "$N_SHARDS"
set -e
SHARD_OUT_DIR=$1
BUCKET_DIR=$2
N_SHARDS=$3

DEST="${BUCKET_DIR}/grm_shards"
mkdir -p "$DEST"

n_copied=0
n_skipped=0
for k in $(seq 1 "$N_SHARDS"); do
  # plink's real output naming is "<out>.grm.bin.<k>", not "<out>.grm.bin"
  src="${SHARD_OUT_DIR}/grm_shard_${k}_of_${N_SHARDS}.grm.bin.${k}"
  dest="${DEST}/grm_shard_${k}_of_${N_SHARDS}.grm.bin.${k}"
  if [ -f "$dest" ]; then
    n_skipped=$((n_skipped + 1))
  else
    cp "$src" "$dest"
    n_copied=$((n_copied + 1))
  fi
done

# one canonical .grm.id, from shard k=1 -- every shard's is identical
cp "${SHARD_OUT_DIR}/grm_shard_1_of_${N_SHARDS}.grm.id" "${DEST}/genome_wide.grm.id"

echo "Persisted ${n_copied} new shard .grm.bin files (${n_skipped} already present) + 1 .grm.id to ${DEST}"

## Next steps

Verify before trusting this run:

1. **Total `.grm.bin` size** should equal `N(N+1)/2 x 4 bytes` (`N` =
   sample count) regardless of `N_SHARDS` -- sum the persisted shard file
   sizes and check against that, as a sanity check that no shard silently
   failed or double-counted rows.
2. **`FID` in the one kept `.grm.id`**: `grm_shard_tool`'s pheno lookup
   keys on the full `(FID, IID)` pair, while `write_grm_pheno()` sets
   `FID = IID = person_id` -- plink's `.grm.id` commonly defaults `FID` to
   `"0"`. Check this file's actual `FID` column before the phenotype
   cross-product step trusts it (see `03_grm_shards/README.md`).

`GRM-pairs`/`grm_shard_tool` (`accumulate`/`merge`) is the next stage after
this, and is out of scope here -- separate step, separate notebook, per
the standing note not to touch that submodule as part of shard
construction. See `04_process_shards/notebooks/remote/grm_shard_processing.ipynb`.